# Notebook for fragmentation rate uncertainty

This notebook execute some Julia scripts that generate a population of ellipsoidals clumps in 3D in order to compute the uncertainty of the fragmentation rate in 3D from a 2D measurement.

In [25]:
include("EllipsoidFragmentation.jl")
include("ProjectionMapper.jl")
include("SimulationIO.jl")

import .EllipsoidFragmentation
import .ProjectionMapper
import .SimulationIO

import Plots
import DataFrames
import Statistics

# Test example to compute FAMILY uncertainties

In [26]:
# Load configuration
config = SimulationIO.load_config("uncertainty_config.toml")
parents = SimulationIO.create_initial_parents(config; seed=42)

# Test differents fragmentation rates
frag_rates = [3.0]
scaling_ratios = SimulationIO.set_scaling_ratios(config)
println("scaling_ratios = $scaling_ratios")

errors = Float64[]
phi2d = Float64[]

for phi in frag_rates
    parents_test = deepcopy(parents)
    gen_test = EllipsoidFragmentation.Generator(parents_test, 
                        fill(phi, length(scaling_ratios)),
                        scaling_ratios,
                        0.5, 1e-3, 5000)
    
    ells = EllipsoidFragmentation.build_population!(gen_test; verbose=false)
    proj_test = ProjectionMapper.Projector(ells)
    projections_test = ProjectionMapper.project_population(proj_test)

    # Compute error
    counts_2d, children_counts_2d, sigma_counts_2d = ProjectionMapper.count_by_level(proj_test.data, projections_test)

    for axis in (:xy, :xz, :yz)
        println("axis $axis :")
        println("\tNumber of parents =\t", counts_2d[axis][0])
        println("\tNumber of children in 3D =\t", counts_2d[axis][1])
        println("\tNumber of children in 2D =\t", children_counts_2d[axis][0])
    end
    max_ratio = prod(scaling_ratios)

    average_n = counts_2d[:xy][0] / counts_2d[:xy][1]
    phi_mes3d = - log(average_n) / log(max_ratio)

    # println(max_ratio)
    average_n = counts_2d[:xy][0] / children_counts_2d[:xy][0]
    sigma_average = sigma_counts_2d[:xy][0] / sqrt(counts_2d[:xy][0])

    phi_mes2d = - log(average_n) / log(max_ratio)
    sigma_phi_mes2d = sigma_average / log(max_ratio) / average_n
    push!(phi2d, phi_mes2d)
    push!(errors, sigma_phi_mes2d)
    
    println("φ = $phi")
    println("φ3D = $phi_mes3d")
    println("φ2D = $phi_mes2d +/- $sigma_phi_mes2d")
end

scaling_ratios = [2.0]


Progress: 100%|█████████████████████████████████████████| Time: 0:05:50
Projecting ellipsoids... 100%|███████████████████████████| Time: 0:00:00


axis xy :
	Number of parents =	1000
	Number of children in 3D =	6596
	Number of children in 2D =	4365
axis xz :
	Number of parents =	1000
	Number of children in 3D =	6596
	Number of children in 2D =	4351
axis yz :
	Number of parents =	1000
	Number of children in 3D =	6596
	Number of children in 2D =	4521
φ = 3.0
φ3D = 2.7215913987753804
φ2D = 2.1259816538547156 +/- 0.21896215605319777


In [ ]:
Plots.plot(frag_rates, phi2d,
    xlabel="φ 3D",
    ylabel="Measured φ 2D",
    marker=:circle,
    linewidth=2,
    legend=false)

Plots.plot!(frag_rates, frag_rates,
    linestyle = :dot,
    legend=false)

## Create and save reference tables in HDF5 file

Not needed to recompute, just here to show how the tables in FAMILY were computed. Values are multiplied by 10,000, rounded, and stored as Int16 to reduce storage requirements. Divide the stored integers by 10,000 to recover the values with a precision of four decimal.
Note: this gives a resolution of 10⁻⁴. Below such resolution, a scaling ratio $R/R_0 \approx 10^{-43}$ would be required to produce a 1% change in the number of fragments produced. A variation of $10^{-4}$ in $\phi$ has a negligible effect on the fragmentation between the galaxy size ($\sim 10^9~AU$) and the first larson core ($\sim 10~AU$).

In [28]:
function compute_phi2d(config, parents, phi3d, scaling_ratio, overlap)
    println("Computing uncertainty with phi3D = $phi3d, ratio = $scaling_ratio, overlap = $overlap")
    parents_test = deepcopy(parents)
    gen_test = EllipsoidFragmentation.Generator(parents_test, 
                        fill(phi3d, 1),
                        [scaling_ratio],
                        overlap, 
                        config.hard_objects, config.max_placement_attempts)
    
    ells = EllipsoidFragmentation.build_population!(gen_test; verbose=false)
    proj_test = ProjectionMapper.Projector(ells)
    projections_test = ProjectionMapper.project_population(proj_test)

    # Compute error
    counts_2d, children_counts_2d, sigma_counts_2d = ProjectionMapper.count_by_level(proj_test.data, projections_test)

    # println(max_ratio)
    average_n = counts_2d[:xy][0] / children_counts_2d[:xy][0]
    sigma_average = sigma_counts_2d[:xy][0] / sqrt(counts_2d[:xy][0])

    phi_mes2d = - log(average_n) / log(scaling_ratio)
    sigma_phi_mes2d = sigma_average / log(scaling_ratio) / average_n

    println("φ = $phi3d")
    println("φ2D = $phi_mes2d +/- $sigma_phi_mes2d")

    return phi_mes2d, sigma_phi_mes2d
end

compute_phi2d (generic function with 1 method)

In [31]:
using HDF5

config = SimulationIO.load_config("uncertainty_config.toml")
resolution = 10000

phi3D = collect(0.0:0.1:3.0) # from 0 to 3 with 0.1 interval
scaling_ratios = collect(1.2:0.2:3.0)  # from 1.2 to 3 with 0.2 interval
overlap = collect(0.5:0.1:1.0)  # from 0.5 to 1 with 0.1 interval

6-element Vector{Float64}:
 0.5
 0.6
 0.7
 0.8
 0.9
 1.0

In [32]:
h5open("projection_uncertainty.h5", "w") do f
    f["phi"] = phi3D
    f["scaling_ratio"] = scaling_ratios
    f["overlap"] = overlap

    # metadata
    f["n_initial_parents"] = config.n_initial_parents
    f["hard_objects"] = config.hard_objects
    f["max_placement_attempts"] = config.max_placement_attempts

    create_dataset(
        f, "phi2d", Int16,
        (length(phi3D), length(scaling_ratios), length(overlap));
        fill_value=-1
    )

    create_dataset(
        f, "sigma_phi2d", UInt16,
        (length(phi3D), length(scaling_ratios), length(overlap));
        fill_value=-1
    )
end

🔢 HDF5.Dataset: (invalid)

In [ ]:
parents = SimulationIO.create_initial_parents(config; seed=42)

h5open("projection_uncertainty.h5", "r+") do f
    for k in eachindex(overlap)
        if k < 3
            for j in eachindex(scaling_ratios)
                for i in eachindex(phi3D)
                    if i > 20
                        # already computed
                        f["phi2d"][i, j, k] != -1 && continue
        
                        phi2d, sigma_phi2d = compute_phi2d(config, parents, phi3D[i], scaling_ratios[j], overlap[k])
        
                        f["phi2d"][i, j, k] = round.(Int16, phi2d .* resolution) 
                        f["sigma_phi2d"][i, j, k] = round.(UInt16, sigma_phi2d .* resolution)
                    end
                end
            end
        end
    end
end

LoadError: UndefVarError: `SimulationIO` not defined in `Main`
Suggestion: check for spelling errors or missing imports.